# Replicación de Hillstrom Conversion

Notebook completo del piloto Two-Model, desde preparación hasta auditoría del código y entorno originales.
Los resultados guardados se cargan en las secciones de modelos. Reentrenar requiere activar explícitamente sus variables de control.


In [1]:
from pathlib import Path
import sys
import pandas as pd

# Permite ejecutar desde la raíz del proyecto o desde src/notebooks.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / 'src/pipeline/prepare_hillstrom.py').is_file()
     and (p / 'data/hillstrom-email/hillstrom.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Abre el notebook desde la carpeta Replicacion o sus subcarpetas.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.pipeline.prepare_hillstrom import FEATURES, prepare_hillstrom
raw_path = PROJECT_ROOT / 'data/hillstrom-email/hillstrom.csv'
raw = pd.read_csv(raw_path)
print(f'Datos originales: {len(raw):,} clientes')
display(raw.groupby('segment').agg(clientes=('conversion', 'size'), compras=('conversion', 'sum')))


Datos originales: 64,000 clientes


,clientes,compras
segment,,
Mens E-Mail,21307,267
No E-Mail,21306,122
Womens E-Mail,21387,189


## Tratamiento y respuesta

`treatment` distingue correo femenino (1) de ausencia de correo (0). `response` indica compra durante las dos semanas posteriores. Se excluye el correo de productos masculinos.

El pipeline codifica las categorías, retira los resultados posteriores de los predictores y elimina filas idénticas en **las ocho características más tratamiento y respuesta**. No se balancean los grupos.


In [2]:
prepared, audit = prepare_hillstrom(raw_path)
print(f'Antes de retirar duplicados: {audit["rows_selected_before_deduplication"]:,}')
print(f'Duplicados retirados: {audit["rows_removed_after_transformation"]:,}')
print(f'Base final: {len(prepared):,} filas y {len(prepared.columns)} columnas')
print(f'Original intacto: {audit["original_unchanged"]}')
print(f'Tamaños coinciden con el artículo: {audit["paper_counts_match"]}')
display(prepared.head())


Antes de retirar duplicados: 42,693
Duplicados retirados: 4,769
Base final: 37,924 filas y 10 columnas
Original intacto: True
Tamaños coinciden con el artículo: True


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,treatment,response
0,10,1,142.44,1,0,1,0,1,1,0
1,6,2,329.08,1,1,0,1,2,0,0
2,7,1,180.65,0,1,1,1,2,1,0
3,2,0,45.34,1,0,2,0,2,1,0
4,6,1,134.83,0,1,1,0,1,1,0


In [3]:
before = pd.DataFrame(audit['before_deduplication']).T
after = pd.DataFrame(audit['after_deduplication']).T
comparison = before.add_suffix('_antes').join(after.add_suffix('_despues'))
display(comparison[['rows_antes', 'rows_despues', 'conversions_antes', 'conversions_despues', 'conversion_rate_antes', 'conversion_rate_despues']])
print('Predictores:', list(FEATURES))
print('Códigos de categorías:', audit['category_mappings'])
print('Archivos guardados:', audit['outputs'])


,rows_antes,rows_despues,conversions_antes,conversions_despues,conversion_rate_antes,conversion_rate_despues
control,21306.0,18926.0,122.0,122.0,0.005726,0.006446
correo_femenino,21387.0,18998.0,189.0,187.0,0.008837,0.009843


Predictores: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel']
Códigos de categorías: {'history_segment': {'1) $0 - $100': 0, '2) $100 - $200': 1, '3) $200 - $350': 2, '4) $350 - $500': 3, '5) $500 - $750': 4, '6) $750 - $1,000': 5, '7) $1,000 +': 6}, 'zip_code': {'Rural': 0, 'Surburban': 1, 'Urban': 2}, 'channel': {'Multichannel': 0, 'Phone': 1, 'Web': 2}}
Archivos guardados: {'prepared_csv': 'C:\\Users\\davo2\\OneDrive\\Documentos\\Desktop\\Ulima\\Cursos\\26-2\\Tesis\\Replicacion\\data\\hillstrom-email\\processed\\Hillstrom_Email_w_conversion.csv', 'removed_rows_csv': 'C:\\Users\\davo2\\OneDrive\\Documentos\\Desktop\\Ulima\\Cursos\\26-2\\Tesis\\Replicacion\\data\\hillstrom-email\\processed\\Hillstrom_Email_w_conversion_duplicados.csv', 'audit_json': 'C:\\Users\\davo2\\OneDrive\\Documentos\\Desktop\\Ulima\\Cursos\\26-2\\Tesis\\Replicacion\\data\\hillstrom-email\\processed\\Hillstrom_Email_w_conversion_auditoria.json'}


## Interpretación de la preparación

La deduplicación reproduce la decisión del código original. El dataset carece de ID individual, de modo que filas idénticas pueden corresponder a personas distintas.

Se conservan ocho predictores y dos columnas de tratamiento y respuesta. Las siguientes secciones incluyen separación, entrenamiento, evaluación y cierre de la replicación.


## Separación de entrenamiento y prueba

Se reserva el 20 % como prueba final y el 80 % como entrenamiento. La semilla es 123 y la estratificación combina tratamiento y respuesta, de modo que se representan los cuatro grupos. Los futuros modelos usarán estas mismas particiones.

Esta celda llama al Python de `.venv` para utilizar scikit-learn sin cambiar el kernel actual. Los índices de trazabilidad se guardan en un archivo aparte.


In [ ]:
import json
import os
import subprocess

project_python = PROJECT_ROOT / '.venv/Scripts/python.exe'
if not project_python.is_file():
    raise FileNotFoundError('No se encontró el Python local en .venv/Scripts/python.exe.')
split_script = PROJECT_ROOT / 'src/pipeline/split_hillstrom.py'
split_env = os.environ.copy()
split_env['PYTHONIOENCODING'] = 'utf-8'
split_run = subprocess.run([str(project_python), str(split_script)], cwd=PROJECT_ROOT, env=split_env, capture_output=True, text=True, encoding='utf-8')
if split_run.returncode != 0:
    raise RuntimeError(split_run.stderr)
print(split_run.stdout)
split_dir = PROJECT_ROOT / 'data/hillstrom-email/processed/splits'
train = pd.read_csv(split_dir / 'train.csv')
test = pd.read_csv(split_dir / 'test.csv')
split_audit = json.loads((split_dir / 'split_audit.json').read_text(encoding='utf-8'))
X_train, y_train, t_train = train.loc[:, list(FEATURES)], train['response'], train['treatment']
X_test, y_test, t_test = test.loc[:, list(FEATURES)], test['response'], test['treatment']
display(pd.DataFrame(split_audit['strata']))
print('Dimensiones predictoras:', X_train.shape, X_test.shape)


## Uso de las particiones

`X_train` contiene únicamente los ocho predictores; `y_train` contiene la respuesta y `t_train` el tratamiento. Los equivalentes de prueba se reservan para evaluar el modelo final. No usar prueba para escoger hiperparámetros ni decidir qué algoritmo entrenar. La estratificación mantiene aproximadamente las proporciones observadas; no crea ni duplica clientes.


## Entrenamiento de Two-Model

Se ajusta un Random Forest a quienes recibieron correo femenino y otro al control. Para cada perfil, el uplift estimado es `P(compra | correo, perfil) - P(compra | sin correo, perfil)`. No se observa el efecto individual real.

Se probaron 16 configuraciones con diez folds estratificados por tratamiento y respuesta, usando solo entrenamiento. Los parámetros ganadores se aplicaron a ambos bosques y se reentrenaron con todo entrenamiento. La bitácora explica las adaptaciones de la métrica y la compatibilidad con scikit-learn.

Los resultados ya están guardados. La siguiente celda los carga sin repetir el entrenamiento. Para repetir la búsqueda completa, cambiar `REENTRENAR_TWO_MODEL` a `True`.


In [ ]:
import json
import os
import subprocess

REENTRENAR_TWO_MODEL = False
two_model_dir = PROJECT_ROOT / 'results/two_model'
if REENTRENAR_TWO_MODEL:
    training_env = os.environ.copy()
    training_env['PYTHONIOENCODING'] = 'utf-8'
    training_env['PYTHONDONTWRITEBYTECODE'] = '1'
    subprocess.run(
        [str(PROJECT_ROOT / '.venv/Scripts/python.exe'), '-u',
         str(PROJECT_ROOT / 'src/pipeline/train_two_model.py'), '--jobs', '4'],
        cwd=PROJECT_ROOT, env=training_env, check=True,
    )
report_path = two_model_dir / 'training_report.json'
if not report_path.is_file():
    raise FileNotFoundError('No hay resultados. Activa REENTRENAR_TWO_MODEL para entrenar.')
two_model_report = json.loads(report_path.read_text(encoding='utf-8'))
two_model_cv = pd.read_csv(two_model_dir / 'cv_results.csv')
two_model_predictions = pd.read_csv(two_model_dir / 'test_predictions.csv')
print('Parámetros seleccionados:', two_model_report['best_parameters'])
print('Filas del bosque tratado:', two_model_report['treated_model_rows'])
print('Filas del bosque de control:', two_model_report['control_model_rows'])
print('UQC medio de validación para selección:', round(two_model_report['mean_validation_uqc'], 4))
print('Modelo guardado en:', two_model_dir / 'two_model.joblib')
display(two_model_cv.sort_values(['mean_validation_uqc', 'candidate'], ascending=[False, True]).head())
display(two_model_predictions.head())


## Interpretación del entrenamiento

Un uplift positivo indica que el modelo estima mayor probabilidad de compra con el correo. Un valor negativo indica que estima menor probabilidad. Estas puntuaciones no prueban qué individuo compró gracias a la campaña.

El UQC mostrado corresponde a la validación usada para escoger parámetros. La referencia aleatoria de esta definición es 1. No representa la evaluación final en prueba y el puntaje ganador puede ser optimista por la selección.

Las predicciones de prueba se generaron después de congelar los parámetros. El siguiente paso es evaluar esas predicciones mediante la curva y las métricas finales, sin volver a seleccionar parámetros con prueba.


## Evaluación final en prueba y comparación con el artículo

Se evalúan los dos bosques ya guardados, sin entrenar ni ajustar parámetros. Las 7.585 filas de prueba se usan para medir desempeño. El UQC compara el área de la curva de diez deciles con la diagonal aleatoria, cuyo valor de referencia es 1.

La comparación con el artículo distingue su promedio entre modelos de folds de nuestro único par reentrenado. Los parámetros también difieren. El diagnóstico con el desempate original se registra aparte y no sustituye la evaluación principal.

Los resultados se cargan por defecto. Activar `REEVALUAR_TWO_MODEL` solo vuelve a evaluar el modelo guardado, sin reentrenarlo.


In [ ]:
import hashlib
import json
import os
import subprocess

REEVALUAR_TWO_MODEL = False
evaluation_dir = PROJECT_ROOT / 'results/two_model/evaluation'
if REEVALUAR_TWO_MODEL:
    evaluation_env = os.environ.copy()
    evaluation_env['PYTHONIOENCODING'] = 'utf-8'
    evaluation_env['PYTHONDONTWRITEBYTECODE'] = '1'
    subprocess.run(
        [str(PROJECT_ROOT / '.venv/Scripts/python.exe'),
         str(PROJECT_ROOT / 'src/pipeline/evaluate_two_model.py')],
        cwd=PROJECT_ROOT, env=evaluation_env, check=True,
    )
test_report = json.loads((evaluation_dir / 'test_metrics.json').read_text(encoding='utf-8'))
for relative in ['results/two_model/two_model.joblib', 'data/hillstrom-email/processed/splits/test.csv']:
    current_hash = hashlib.sha256((PROJECT_ROOT / relative).read_bytes()).hexdigest()
    if current_hash != test_report['protected_sha256'][relative.replace('/', os.sep)]:
        raise RuntimeError('Cambió el modelo o la prueba. Vuelve a evaluar antes de usar estos resultados.')
test_curve = pd.read_csv(evaluation_dir / 'test_decile_curve.csv')
paper_comparison = pd.read_csv(evaluation_dir / 'paper_comparison.csv')
print('Filas de prueba:', test_report['test_rows'])
print('UQC de prueba:', test_report['test_uqc'])
print('UQC del artículo, promedio publicado:', test_report['paper_reference']['table_5_test_uqc_mean'])
print('AUUC relativa:', round(test_report['test_auuc_relative_decile_curve'], 7))
print('Área Qini por encima de la diagonal:', round(test_report['test_qini_area_above_linear_random_baseline'], 7))
display(paper_comparison)
display(test_curve[['fraction_selected_per_group', 'treated_selected', 'control_selected', 'cumulative_gain_normalized']])


## Lectura de los resultados de prueba

El UQC de **1,3686** está por encima de la referencia 1, pero por debajo de **2,0011** reportado en el artículo. Es una comparación descriptiva de puntuaciones, no una prueba de significancia. La curva tampoco domina a la selección aleatoria en todos los deciles.

El indicador acumulado al seleccionar el 20 % es **0,240 p.p.**, frente a aproximadamente 0,410 p.p. en el artículo. Se divide la ganancia incremental por el total tratado de prueba, no por los tratados seleccionados. La coincidencia de 0,370 p.p. al seleccionar a todos no demuestra que el ranking sea igual.

El desempate por respuesta del código original produce el mismo UQC y la misma curva en este conjunto de prueba. No explica la diferencia observada frente al artículo. No se ha identificado una causa única de esa diferencia y no se ajustan parámetros usando estos resultados.

![Curva Qini en prueba](../../results/two_model/evaluation/test_qini_curve.png)

La tabla completa, las definiciones y las limitaciones están en `BITACORA.md` y en `results/two_model/evaluation`.


## Two-Model con los parámetros exactos publicados

Se fija la configuración de la tabla A1, con 100 árboles, profundidad 25 y mínimo de 30 registros por hoja. Se guardan un par final reentrenado y los diez pares de los folds. Para comparar con la agregación del artículo se promedian las diez métricas de prueba, sin elegir el mejor fold ni promediar scores.

Esta variante se realiza después de haber observado la evaluación anterior de prueba. Se registra como una comparación exploratoria de replicación, no como una nueva selección independiente del mejor modelo. La variante utiliza parámetros publicados aunque no ganaron nuestra validación de entrenamiento.


In [ ]:
import hashlib
import json
import os
import subprocess

REPETIR_CONFIGURACION_ARTICULO = False
paper_dir = PROJECT_ROOT / 'results/two_model_paper_config'
if REPETIR_CONFIGURACION_ARTICULO:
    paper_env = os.environ.copy()
    paper_env['PYTHONIOENCODING'] = 'utf-8'
    paper_env['PYTHONDONTWRITEBYTECODE'] = '1'
    subprocess.run([str(PROJECT_ROOT / '.venv/Scripts/python.exe'), '-u',
                    str(PROJECT_ROOT / 'src/pipeline/replicate_two_model_paper.py'), '--overwrite'],
                   cwd=PROJECT_ROOT, env=paper_env, check=True)
paper_report = json.loads((paper_dir / 'experiment_report.json').read_text(encoding='utf-8'))
for name, expected in paper_report['model_sha256'].items():
    if hashlib.sha256((paper_dir / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('El archivo de modelos cambió respecto del experimento registrado.')
print('Parámetros publicados:', paper_report['parameters'])
print('UQC de validación:', round(paper_report['validation_uqc_mean'], 5))
print('UQC del par final:', paper_report['test_final_refit_uqc'])
print('UQC medio de diez folds:', round(paper_report['test_fold_uqc_mean'], 4))
print('Comparación exploratoria tras la primera evaluación de prueba.')
display(pd.read_csv(paper_dir / 'comparison.csv'))
display(pd.read_csv(paper_dir / 'fold_test_metrics.csv'))


## Resultado de la variante

El modelo final pasa de **1,3686 a 1,6450** de UQC en este conjunto de prueba. El promedio de los diez modelos de folds es **1,3984**, frente a 2,0011 publicado. Son unidades de evaluación distintas y se reportan separadamente.

El resultado de validación con los parámetros publicados sigue siendo inferior al del ganador previo. La diferencia observada en prueba no demuestra una mejora generalizable. Esta variante se conserva para la replicación y no sustituye los resultados anteriores.

![Comparación de curvas](../../results/two_model_paper_config/comparison_qini.png)


## Cierre técnico de la replicación

Los métodos originales se ejecutaron en Python 3.8.10 con scikit-learn 1.0.2, NumPy 1.22.4 y pandas 1.4.2. La preparación, las particiones y las puntuaciones coinciden con nuestra implementación.

El par final obtiene UQC 1,6450 y el promedio de folds 1,3984. La cifra publicada de 2,0011 no se reprodujo. La comparación posterior a revisar prueba es exploratoria.


In [ ]:
import json
closure_dir = PROJECT_ROOT / 'results/replication_closure'
closure = json.loads((closure_dir / 'closure_audit.json').read_text(encoding='utf-8'))
print('Entorno de referencia:', closure['environment'])
print('UQC del par final:', closure['original_final_refit_uqc'])
print('UQC medio de folds:', closure['original_fold_mean_uqc'])
print('Cifra publicada reproducida:', closure['exact_published_result_reproduced'])
display(pd.read_csv(closure_dir / 'audit_checklist.csv'))
